In [0]:
%sql
select *,_metadata.file_name as source_file,current_time() as ingestion_time from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/orders/",
    format => "json"
)

In [0]:
%sql
DESCRIBE QUERY 
select *,_metadata.file_name as source_file,current_time() as ingestion_time from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/orders/",
    format => "json"
) LIMIT 0;


In [0]:
%sql
CREATE OR REPLACE TABLE orders_bronze AS(
    select *,_metadata.file_name as source_file,current_timestamp() as ingestion_time from read_files(
    "/Volumes/databricks_labs/default/lab_data/week3/orders/",
    format => "json"
)
);

SELECT * FROM orders_bronze;

In [0]:
%sql
DESCRIBE TABLE orders_bronze;

In [0]:
%sql
CREATE or REPLACE  TABLE orders_silver AS(
    SELECT customer_id,notifications,order_id,CAST(order_timestamp AS TIMESTAMP),source_file,ingestion_time,to_date(CAST(order_timestamp AS TIMESTAMP)) as order_date 
    FROM orders_bronze
    where order_id is not null and customer_id is not null
);

select * from orders_silver;

In [0]:
%sql
CREATE OR REPLACE VIEW orders_gold AS(
    select order_date,count(order_id) as total_orders,count(distinct customer_id) as unique_customers
    from orders_silver
    group by order_date
    order by order_date
);

select * from orders_gold

In [0]:
%sql
select * from databricks_labs.default.orders_bronze_st

In [0]:
%sql
SELECT COUNT(*) AS silver_count
FROM databricks_labs.default.orders_silver_st;

In [0]:
%sql
SELECT 'Bronze' AS layer, COUNT(*) AS records
FROM databricks_labs.default.orders_bronze_st

UNION ALL

SELECT 'Silver', COUNT(*)
FROM databricks_labs.default.orders_silver_st

UNION ALL

SELECT 'Gold', SUM(total_orders)
FROM databricks_labs.default.orders_gold_st;

In [0]:
%sql
SELECT TRY_CAST('100$' AS INT);

In [0]:
%sql
--what happens when fail update condition comes true
SELECT COUNT(*) AS total_records
FROM databricks_labs.default.orders_silver_fail_st;

In [0]:
%sql
drop table databricks_labs.default.orders_bronze_fail_st;

In [0]:
%sql
drop table databricks_labs.default.orders_silver_fail_st;